In [2]:
import requests
import json
import pandas as pd
import re
import time

def get_naver_search_results(query):
    # 실습 파일에 기록된 본인의 API 키를 입력하세요
    headers = {
        'X-Naver-Client-Id' : 'qlURw5tgE7CmccdW37fe',
        'X-Naver-Client-Secret' : 'FOGoT6pEeT'
    }
    
    all_items = []
    # 네이버 API는 최대 1,000건(100건씩 10번)까지 조회가 가능합니다.
    for i in range(10):
        start = (i * 100) + 1
        # 블로그 검색(blog.json) 주소를 사용합니다.
        url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}&sort=date"
        
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = json.loads(response.text)
            items = data.get('items', [])
            if not items:
                break
            all_items.extend(items)
            time.sleep(0.1) # 서버 부하 방지
        else:
            print(f"Error Code: {response.status_code}")
            break
            
    return all_items

# 1. 두 키워드에 대해 각각 수집을 진행합니다.
print("🔎 '렉스트림' 검색 중...")
ko_results = get_naver_search_results("렉스트림")

print("🔎 'rextreme' 검색 중...")
en_results = get_naver_search_results("rextreme")

# 2. 데이터 합치기 및 중복 제거
total_results = ko_results + en_results
df = pd.DataFrame(total_results)

if not df.empty:
    # 링크(link)를 기준으로 중복된 포스팅 제거 (두 키워드 모두 쓴 글 대비)
    df = df.drop_duplicates(subset=['link']).reset_index(drop=True)

    # 3. 텍스트 정제 (HTML 태그 제거)
    clean_re = re.compile('<.*?>')
    df['title'] = df['title'].apply(lambda x: re.sub(clean_re, '', x))
    df['description'] = df['description'].apply(lambda x: re.sub(clean_re, '', x))

    # 4. 저장
    df.to_csv('naver_rextreme_blog2.csv', index=False, encoding='utf-8-sig')
    print(f"\n✅ 수집 완료! 중복 제거 후 총 {len(df)}건의 데이터를 저장했습니다.")
else:
    print("검색 결과가 없습니다.")

🔎 '렉스트림' 검색 중...
🔎 'rextreme' 검색 중...

✅ 수집 완료! 중복 제거 후 총 242건의 데이터를 저장했습니다.
